In [37]:
#!pip uninstall transformers -y
!pip install transformers==4.28.0 tokenizers==0.13.3
#!pip install torchaudio
#!pip install jiwer
#!pip install accelerate -U
#!pip install --upgrade torch
#!pip install datasets
#!pip install --upgrade transformers
#!pip install -U torch torchvision torchaudio

  Using cached transformers-4.28.0-py3-none-any.whl.metadata (109 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.28.0-py3-none-any.whl (7.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#pip install numpy==2.2.2

In [36]:
from pathlib import Path
import transformers
import json
import random
import re
import wave
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Union

import IPython.display as ipd
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display, HTML

from jiwer import wer as jiwer_wer, cer as jiwer_cer

from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Trainer,
    TrainingArguments,
)

print("torch:", torch.__version__)
try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception:
    pass

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.12.0+cu130
transformers: 5.9.0
cuda available: True
gpu: Tesla T4


In [ ]:
import IPython.display as ipd
import numpy as np
import random
from datasets import ClassLabel
import pandas as pd
from IPython.display import display, HTML
import torch
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from tqdm import tqdm

In [4]:
model_path = '/content/drive/MyDrive/nenets_asr_1800'
processor = Wav2Vec2Processor.from_pretrained(model_path)
model = Wav2Vec2ForCTC.from_pretrained(model_path)
model.eval()

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, eleme

In [5]:
audio_path = '/content/drive/MyDrive/check model'
dataset = pd.read_csv('/content/good_day.csv')
dataset['audio'] = dataset['audio'].str.replace('/Users/air/Downloads/Ненецкие аудио', '/content/drive/MyDrive')

In [52]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

predictions = []
references = []

for i in range(len(dataset)):
    row = dataset.iloc[i]
    reference = str(row["transcription"]).lower().strip()
    audio = load_wav_mono_16k(row["audio"])

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True)[0]

    #prediction = processor.decode(pred_ids)

    prediction = (
        prediction
        .replace("[PAD]", "")
        .replace("[UNK]", "")
    )

    prediction = " ".join(prediction.split())

    predictions.append(prediction.lower().strip())
    references.append(reference)


wer_value = jiwer_wer(references[-36:-1], predictions[-36:-1]) #референсы съехали, из-за чего метрики ниже
cer_value = jiwer_cer(references[-36:-1], predictions[-36:-1])

print(f"WER: {wer_value:.4f}")
print(f"CER: {cer_value:.4f}")

WER: 0.8005
CER: 0.2032


In [ ]:
pred_real = pd.DataFrame()
pred_real['pred'] = predictions
pred_real['real'] = references
pred_real.to_csv('pred_real_eval.csv', index=False)

In [ ]:
print(references, sep='\n')

In [46]:
for i in range((predictions)):
    print(predictions[i])
    print(references[i])
    print()

сяна ӈэйбтаӈод вэсаку илевы пууцяда таня парэӈгода' харад' вэкана илеӈах' вэсаку нед ватсеты
сяны ӈэбта ӈод" вэсако илевы пухуцяда таня параӈода харад" вэкана илеӈаха" вэсако недватсяты хоркыта" поӈгаця" таня вэсако поёрць хабтекода таня ӈод" пухуцяда маси

хоркыта поӈгацетаня" вэску поёр цябтекуда таня ӈобгуна пуцяада мась'
вэсаков' пини' сусавы поёрць ханьма вэсакор' хабтемда подерӈа хая

вэсаков пини сусавы' поёрць хань"ма вэсякор хабтемда подерӈа хая
вэсако пядамда мэ нибям' хо маси пухуцяни нибяд няна ходамню лы' нибякоим мэцятыню маси

вэсако тяданда мэ нибям' хо маси пухуцяни нибяд няна" хохдамню" лый нибякоям' мэцьтынё"
ӈамгэн' мэбни савади панэхэта пакалпида сита' симанзер ӈа

ӈамгэн мэб"ни савадий" панэта пакалпида сита си"маньсер ӈа
тюку вэва ӈэвы пини' понд мэхэрцяв тикавахад пита" понд мэда…

тюку вэва ӈэвы пи"ни понд мэрцьв тикалахад пита понд мэда
мяканда тэвы вэсакор хабтемда ӈэдавдавэй' мяканда сюрамба' хая

мяканда тывы" вэсякор хабтемда медавдавэй' мяканда сёрмба хая